In [3]:
# Importacoes Necessarias

from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import *

StatementMeta(, 2497e580-d826-427e-839c-50cb0cb40800, 5, Finished, Available, Finished, False)

In [2]:
# Lemos a tabela mestre no Lakehouse para pegar as moedas ativas de forma dinâmica

df_moedas = spark.sql("SELECT * FROM masterdata_moedas")

# Coletamos os dados e criamos uma lista (Array Python) contendo apenas as siglas

lista_moedas = [row['Moeda'] for row in df_moedas.collect()]
print(lista_moedas)

StatementMeta(, 45b53433-199a-497f-9737-ddd53a762a5f, 4, Finished, Available, Finished, False)

['SEK', 'USD', 'AUD', 'NOK', 'CAD', 'GBP', 'CHF', 'DKK', 'EUR', 'JPY']


In [8]:
# Definimos a janela de tempo da extração (Padrão americano exigido pela API: MM-DD-YYYY)

data_inicial = '01-01-2020'
data_final = datetime.now().strftime('%m-%d-%Y')

# Parâmetros de paginação da API OData do Banco Central

top = 100

for moeda in lista_moedas:

    # A cada nova moeda iterada, garantimos que o skip e a lista de dados sejam zerados

    skip = 0
    todos_dados = []

    while True:

        url = (
            "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
            f"CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
            f"@moeda='{moeda}'&@dataInicial='{data_inicial}'&@dataFinalCotacao='{data_final}'"
            f"&$top={top}&$skip={skip}&$filter=tipoBoletim%20eq%20'Fechamento'&$format=json&$select=cotacaoCompra,dataHoraCotacao"
        )

        response = requests.get(url)

        # Boa Prática: Usar .get('value', []) evita que o código quebre (KeyError) 
        # Caso a API retorne um erro temporário e não traga a chave 'value' no JSON

        dados = response.json()['value']

        # Condição de parada: Se a lista voltar vazia, significa que acabaram as páginas

        if not dados:
            break

        # Acumula a página atual na lista principal e avança o ponteiro do 'skip'

        todos_dados.extend(dados)
        skip += top

if todos_dados:
    # Cria um DataFrame PySpark a partir da lista de dicionários e adiciona a coluna identificadora

    df = spark.createDataFrame(todos_dados)\
        .withColumn('moeda', lit(moeda))

    # Transforma o formato de data (MM-DD-YYYY) para um padrão amigável para pastas/arquivos (YYYYMMDD)

    data_inicial_path = datetime.strptime(data_inicial, "%m-%d-%Y").strftime("%Y%m%d")
    data_final_path = datetime.strptime(data_final, "%m-%d-%Y").strftime("%Y%m%d")

    # Constrói o caminho final do arquivo Parquet no Lakehouse (OneLake)

    path = (
        f"Files/Cotacoes/Novos/"
        f"{moeda}-"
        f"{data_inicial_path}_"
        f"{data_final_path}"
        f".parquet"
    )

    # Salva o arquivo sobrescrevendo caso ele já exista no caminho especificado
    
    df.write.mode('overwrite').parquet(path)

StatementMeta(, 2497e580-d826-427e-839c-50cb0cb40800, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5bd8055c-39df-4ec0-b8b3-ac70be2df508)